# 01 — Data Acquisition

Downloads all data from the [Anthropic Economic Index](https://huggingface.co/datasets/Anthropic/EconomicIndex) on HuggingFace,
along with O\*NET occupational data and BLS employment statistics.

**Data sources:**
- Anthropic Economic Index (4 releases: 2025-03, 2025-09, 2026-01, 2026-03)
- O\*NET task statements and SOC structure
- BLS employment and wage data
- Labor market impact scores (job exposure, task penetration)

In [1]:
import sys
sys.path.insert(0, "..")

import logging
import pandas as pd

from src.data import (
    download_all,
    load_onet_tasks,
    load_soc_structure,
    load_collaboration_by_task,
    load_unified_release,
    load_wage_data,
    load_bls_employment,
    load_job_exposure,
    load_task_penetration,
    RELEASE_FILES,
)

logging.basicConfig(level=logging.INFO)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

## Download all datasets

In [2]:
paths = download_all()
print(f"Downloaded {len(paths)}/{len(RELEASE_FILES)} files")
for name, path in paths.items():
    size_mb = path.stat().st_size / 1e6
    print(f"  {name:30s} {size_mb:6.1f} MB")

INFO:src.data:Downloading release_2025_03_27/automation_vs_augmentation_v1.csv → release_2025_03_27_automation_vs_augmentation_v1.csv
INFO:src.data:Downloading release_2025_03_27/automation_vs_augmentation_v2.csv → release_2025_03_27_automation_vs_augmentation_v2.csv


INFO:src.data:Downloading release_2025_09_15/data/intermediate/aei_raw_1p_api_2025-08-04_to_2025-08-11.csv → release_2025_09_15_data_intermediate_aei_raw_1p_api_2025-08-04_to_2025-08-11.csv


INFO:src.data:Downloading release_2026_01_15/data/intermediate/aei_raw_1p_api_2025-11-13_to_2025-11-20.csv → release_2026_01_15_data_intermediate_aei_raw_1p_api_2025-11-13_to_2025-11-20.csv


INFO:src.data:Downloading release_2026_03_24/data/aei_raw_1p_api_2026-02-05_to_2026-02-12.csv → release_2026_03_24_data_aei_raw_1p_api_2026-02-05_to_2026-02-12.csv


Downloaded 19/19 files
  task_mappings_v1                  0.5 MB
  collab_aggregate_v1               0.0 MB
  onet_tasks                        3.6 MB
  soc_structure                     0.1 MB
  bls_employment                    0.0 MB
  wage_data                         0.1 MB
  task_pct_v1                       0.5 MB
  task_pct_v2                       0.4 MB
  collab_by_task                    0.6 MB
  collab_aggregate_v1_r2            0.0 MB
  collab_aggregate_v2               0.0 MB
  aei_claude_ai_2025_09            18.9 MB
  aei_api_2025_09                   7.0 MB
  aei_claude_ai_2026_01            94.1 MB
  aei_api_2026_01                  41.5 MB
  aei_claude_ai_2026_03            96.0 MB
  aei_api_2026_03                  44.0 MB
  job_exposure                      0.0 MB
  task_penetration                  1.9 MB


## Inspect O\*NET task structure

O\*NET provides ~19,500 task statements mapped to ~974 occupations via SOC codes.
Each task belongs to one or more occupations, and the AEI data tells us which
tasks appear in Claude conversations.

In [3]:
onet = load_onet_tasks()
print(f"Task statements: {len(onet):,}")
print(f"Unique tasks: {onet['task'].nunique():,}")
print(f"Occupations (O*NET-SOC): {onet['onet_soc_code'].nunique()}")
print(f"Occupations (6-digit SOC): {onet['onet_soc_code'].str[:7].nunique()}")
print()
onet.head()

Task statements: 19,530
Unique tasks: 18,429
Occupations (O*NET-SOC): 974
Occupations (6-digit SOC): 775



,onet_soc_code,title,task_id,task,task_type,incumbents_responding,date,domain_source
0,11-1011.00,Chief Executives,8823,Direct or coordinate an organization's financi...,Core,87.0,07/2014,Incumbent
1,11-1011.00,Chief Executives,8831,Appoint department heads or managers and assig...,Core,87.0,07/2014,Incumbent
2,11-1011.00,Chief Executives,8825,Analyze operations to evaluate performance of ...,Core,87.0,07/2014,Incumbent
3,11-1011.00,Chief Executives,8826,"Direct, plan, or implement policies, objective...",Core,87.0,07/2014,Incumbent
4,11-1011.00,Chief Executives,8827,"Prepare budgets for approval, including those ...",Core,87.0,07/2014,Incumbent


## Inspect collaboration mode data (2025-03 release)

The first per-task collaboration breakdown. Each task gets a distribution across
five interaction modes:

| Mode | Type | Description |
|------|------|-------------|
| **directive** | Automation | AI provides complete autonomous execution |
| **feedback_loop** | Automation | Iterative AI-driven refinement |
| **validation** | Augmentation | Human validates AI output |
| **task_iteration** | Augmentation | Human iterates on AI drafts |
| **learning** | Augmentation | Human learns from AI insights |

In [4]:
collab = load_collaboration_by_task()
print(f"Tasks with collaboration data: {len(collab)}")
print(f"\nMode distributions (mean across tasks):")
for mode in ["directive", "feedback_loop", "task_iteration", "validation", "learning", "filtered"]:
    print(f"  {mode:20s} {collab[mode].mean():.1%}")
print()
collab.sort_values("directive", ascending=False).head(10)

Tasks with collaboration data: 3364

Mode distributions (mean across tasks):
  directive            18.0%
  feedback_loop        1.8%
  task_iteration       14.5%
  validation           0.9%
  learning             19.4%
  filtered             45.4%



,task_name,feedback_loop,directive,task_iteration,validation,learning,filtered
1931,"monitor and observe experiments, recording pro...",0.0,1.000000,0.000000,0.0,0.0,0.000000
506,compile cue words and phrases and cue announce...,0.0,0.870968,0.000000,0.0,0.0,0.129032
2907,return borrowed or rented items when productio...,0.0,0.857143,0.000000,0.0,0.0,0.142857
3140,"take dictation using shorthand, a stenotype ma...",0.0,0.851852,0.000000,0.0,0.0,0.148148
3202,transcribe dictation for a variety of medical ...,0.0,0.850000,0.000000,0.0,0.0,0.150000
2461,"prepare, store, and retrieve classification an...",0.0,0.842105,0.000000,0.0,0.0,0.157895
811,"create photographic recordings of information,...",0.0,0.840000,0.000000,0.0,0.0,0.160000
3219,transpose music from one voice or instrument t...,0.0,0.797297,0.000000,0.0,0.0,0.202703
730,convert various types of files for printing or...,0.0,0.797101,0.166667,0.0,0.0,0.036232
1972,"observe and evaluate students' performance, be...",0.0,0.790698,0.139535,0.0,0.0,0.069767


## Inspect unified-schema releases (2025-09 onward)

Later releases use a long-format schema with facets for different analysis dimensions.
The key facet for us is `onet_task::collaboration`, which gives per-task collaboration
mode shares comparable to the 2025-03 data.

In [5]:
for release in ["2025_09", "2026_01", "2026_03"]:
    df = load_unified_release(release)
    facets = df["facet"].unique()
    collab_rows = (df["facet"] == "onet_task::collaboration").sum()
    date_range = f"{df['date_start'].iloc[0]} to {df['date_end'].iloc[0]}"
    print(f"Release {release}: {len(df):,} rows, {len(facets)} facets, "
          f"{collab_rows:,} collaboration rows, period: {date_range}")

Release 2025_09: 100,062 rows, 7 facets, 14,454 collaboration rows, period: 2025-08-04 to 2025-08-11
Release 2026_01: 458,778 rows, 34 facets, 16,778 collaboration rows, period: 2025-11-13 to 2025-11-20
Release 2026_03: 425,257 rows, 34 facets, 17,530 collaboration rows, period: 2026-02-05 to 2026-02-12


## Inspect wage and employment data

In [6]:
wages = load_wage_data()
print(f"Wage records: {len(wages)}")
print(f"Median salary range: ${wages['mediansalary'].min():,.0f} – ${wages['mediansalary'].max():,.0f}")
print()

emp = load_bls_employment()
print(f"BLS employment by major group: {len(emp)} groups")
print(f"Total employment: {emp['bls_distribution'].sum():,.0f}")
print()

exposure = load_job_exposure()
print(f"Job exposure scores: {len(exposure)} occupations")
print(f"Exposure range: {exposure['observed_exposure'].min():.4f} – {exposure['observed_exposure'].max():.4f}")

Wage records: 1090
Median salary range: $16 – $208,000

BLS employment by major group: 22 groups
Total employment: 151,853,870

Job exposure scores: 756 occupations
Exposure range: 0.0000 – 0.7451


## Build the occupation panel

This is the core dataset: collaboration modes aggregated to the occupation level
across all four releases, merged with wage and exposure data.

In [7]:
from src.data import build_occupation_panel

panel = build_occupation_panel()
print(f"Occupation panel: {panel.shape[0]} rows × {panel.shape[1]} columns")
print(f"Unique occupations: {panel['soc_code'].nunique()}")
print(f"Releases: {sorted(panel['release'].unique())}")
print(f"Wage coverage: {panel['mediansalary'].notna().mean():.1%}")
print(f"Exposure coverage: {panel['observed_exposure'].notna().mean():.1%}")
print()
panel.head()

Occupation panel: 2894 rows × 16 columns
Unique occupations: 633
Releases: ['2025_03', '2025_09', '2026_01', '2026_03']
Wage coverage: 100.0%
Exposure coverage: 83.3%



,soc_code,title,release,automation_share,augmentation_share,task_count,task_iteration,feedback_loop,directive,learning,validation,mediansalary,jobzone,chanceauto,jobforecast,observed_exposure
0,11-1011,Chief Executives,2025_03,0.225997,0.569004,9,0.259662,0.005873,0.220124,0.304879,0.004463,189600.0,5,1.5,16800,0.0333
1,11-1011,Chief Executives,2025_09,0.133584,0.246049,7,0.282627,0.043783,0.297101,0.361559,0.075673,189600.0,5,1.5,16800,0.0333
2,11-1011,Chief Executives,2026_01,0.182297,0.383785,5,0.292837,0.053155,0.286110,0.288241,0.087847,189600.0,5,1.5,16800,0.0333
3,11-1011,Chief Executives,2026_03,0.110809,0.248557,8,0.307170,0.054923,0.258876,0.294573,0.091614,189600.0,5,1.5,16800,0.0333
4,11-1011,Chief Sustainability Officers,2025_03,0.160171,0.583941,5,0.482186,0.000000,0.160171,0.101754,0.000000,189600.0,5,1.5,16800,0.0333


---

**Next:** [02_exploration.ipynb](02_exploration.ipynb) — Exploratory analysis of automation patterns across occupations.